# Trabajo Práctico Integrador

**Fernanda Andrea Fava** · Seminario de Actualización · IFTS N.º 18

## Clasificación zero-shot de mensajes de clientes

Objetivo: comparar modelos preentrenados de Hugging Face para clasificar mensajes breves de clientes en español, sin entrenar modelos propios.

## 1. Problema

Clasificar automáticamente mensajes breves de clientes según el motivo principal del contacto.

Categorías iniciales:
- faltante de producto
- precio o facturación
- calidad del producto
- atención al cliente
- entrega o demora


## 2. Modelos candidatos

Se compararán al menos estos dos modelos zero-shot multilingües:

1. `MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli`
2. `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`

La comparación buscará evaluar calidad de clasificación y eficiencia.

## 3. Pregunta e hipótesis

**Pregunta:** ¿qué modelo ofrece el mejor equilibrio entre calidad de clasificación y eficiencia para mensajes breves en español?

**Hipótesis inicial:** el modelo MiniLM, al ser más liviano, debería ofrecer menor costo computacional; mDeBERTa podría lograr mejores resultados en casos ambiguos o complejos.

In [16]:
print("kernel funcionando")

kernel funcionando


In [17]:
CATEGORIAS = [
    "problema de stock o producto faltante",
    "problema de precio o facturación",
    "problema de calidad del producto",
    "problema de atención al cliente",
    "problema de entrega o demora",
]

MODELO_MINILM = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
MODELO_MDEBERTA = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

print("Variables cargadas")

Variables cargadas


## 4. Set de prueba

Se usan los mismos mensajes y categorías para todos los modelos y métodos de inferencia, de modo que la comparación sea consistente.

El conjunto contiene **8 frases**. Los casos 7 y 8 fueron incluidos intencionalmente como **casos adversariales**: contienen más de un motivo y obligan al modelo a elegir una única categoría principal (`multi_label=False`). El objetivo es observar cómo responde ante mensajes ambiguos y con señales que compiten entre sí.

- Caso 7: `El pedido llegó tarde y además faltaba un producto` combina entrega/demora y faltante de producto.
- Caso 8: `Me cobraron bien, pero la atención fue pésima` menciona facturación de forma positiva y atención de forma negativa.

Estos ejemplos no buscan facilitarle la tarea al modelo: están pensados para poner a prueba sus límites.

In [18]:
CASOS_PRUEBA = [
    (
        "Me cobraron dos veces el mismo producto",
        "problema de precio o facturación"
    ),
    (
        "No había leche ni yogur en la góndola",
        "problema de stock o producto faltante"
    ),
    (
        "El pollo tenía mal olor cuando abrí el paquete",
        "problema de calidad del producto"
    ),
    (
        "La persona que me atendió resolvió mi problema muy rápido",
        "problema de atención al cliente"
    ),
    (
        "El pedido llegó dos horas tarde",
        "problema de entrega o demora"
    ),
    (
        "No había stock y nadie sabía decirme cuándo reponían",
        "problema de stock o producto faltante"
    ),
    (
        "El pedido llegó tarde y además faltaba un producto",
        "problema de entrega o demora"
    ),
    (
        "Me cobraron bien, pero la atención fue pésima",
        "problema de atención al cliente"
    ),
]


## 5. Prueba con `pipeline`

Primero se ejecutará la misma tarea zero-shot con ambos modelos usando `transformers.pipeline`.

In [19]:
from transformers import pipeline
import time
import pandas as pd

def evaluar_modelo_pipeline(nombre_modelo, casos, categorias):
    clasificador = pipeline(
        "zero-shot-classification",
        model=nombre_modelo
    )

    resultados = []

    for texto, categoria_esperada in casos:
        inicio = time.perf_counter()

        salida = clasificador(
            texto,
            candidate_labels=categorias,
            multi_label=False,
            hypothesis_template="Este mensaje trata sobre {}."
        )

        duracion = time.perf_counter() - inicio

        categoria_predicha = salida["labels"][0]
        score = salida["scores"][0]

        resultados.append({
            "modelo": nombre_modelo,
            "metodo": "pipeline",
            "texto": texto,
            "esperada": categoria_esperada,
            "predicha": categoria_predicha,
            "score": score,
            "tiempo_segundos": duracion,
            "acierto": categoria_predicha == categoria_esperada
        })

    return pd.DataFrame(resultados)

In [20]:
resultados_minilm = evaluar_modelo_pipeline(
    MODELO_MINILM,
    CASOS_PRUEBA,
    CATEGORIAS
)

resultados_minilm

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,modelo,metodo,texto,esperada,predicha,score,tiempo_segundos,acierto
0,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,Me cobraron dos veces el mismo producto,problema de precio o facturación,problema de precio o facturación,0.427221,0.031676,True
1,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,No había leche ni yogur en la góndola,problema de stock o producto faltante,problema de stock o producto faltante,0.339001,0.037422,True
2,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,El pollo tenía mal olor cuando abrí el paquete,problema de calidad del producto,problema de entrega o demora,0.462014,0.037793,False
3,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,La persona que me atendió resolvió mi problema...,problema de atención al cliente,problema de atención al cliente,0.421868,0.036906,True
4,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,El pedido llegó dos horas tarde,problema de entrega o demora,problema de atención al cliente,0.437472,0.029877,False
5,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,No había stock y nadie sabía decirme cuándo re...,problema de stock o producto faltante,problema de stock o producto faltante,0.415952,0.031877,True
6,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,El pedido llegó tarde y además faltaba un prod...,problema de entrega o demora,problema de stock o producto faltante,0.389473,0.031001,False
7,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,pipeline,"Me cobraron bien, pero la atención fue pésima",problema de atención al cliente,problema de precio o facturación,0.276240,0.031945,False


### 5.1 ¿Qué devuelve `pipeline`?

Además de medir el resultado, es importante inspeccionar el **tipo de dato** que devuelve la inferencia local. `pipeline` devuelve un diccionario de Python. Dentro de ese diccionario, las etiquetas y los puntajes se consultan por clave.

La celda siguiente muestra el tipo del objeto completo y cómo se accede a la etiqueta y al score principales.

In [ ]:
from transformers import pipeline

clasificador_tipo = pipeline(
    "zero-shot-classification",
    model=MODELO_MINILM
)

salida_pipeline = clasificador_tipo(
    "Me cobraron dos veces el mismo producto",
    candidate_labels=CATEGORIAS,
    multi_label=False,
    hypothesis_template="Este mensaje trata sobre {}."
)

print("Tipo de salida:", type(salida_pipeline))
print("Tipo de labels:", type(salida_pipeline["labels"]))
print("Tipo del primer score:", type(salida_pipeline["scores"][0]))
print("Etiqueta principal:", salida_pipeline["labels"][0])
print("Score principal:", salida_pipeline["scores"][0])

**Lectura del valor de retorno**

- `type(salida_pipeline)` muestra que el resultado es un `dict`.
- La etiqueta principal se obtiene con `salida_pipeline["labels"][0]`.
- El puntaje principal se obtiene con `salida_pipeline["scores"][0]`.
- En este caso se accede a la información mediante **claves de diccionario**.

## 5.2 Evaluación del segundo modelo: mDeBERTa

Se evalúa `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` con exactamente los mismos mensajes, categorías y configuración utilizados con MiniLM. Mantener las mismas condiciones permite realizar una comparación justa entre ambos modelos.

In [21]:
resultados_mdeberta = evaluar_modelo_pipeline(
    MODELO_MDEBERTA,
    CASOS_PRUEBA,
    CATEGORIAS
)

resultados_mdeberta

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

,modelo,metodo,texto,esperada,predicha,score,tiempo_segundos,acierto
0,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,Me cobraron dos veces el mismo producto,problema de precio o facturación,problema de precio o facturación,0.754125,2.029857,True
1,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,No había leche ni yogur en la góndola,problema de stock o producto faltante,problema de calidad del producto,0.259265,2.134339,False
2,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,El pollo tenía mal olor cuando abrí el paquete,problema de calidad del producto,problema de calidad del producto,0.975786,2.284398,True
3,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,La persona que me atendió resolvió mi problema...,problema de atención al cliente,problema de atención al cliente,0.998904,2.144052,True
4,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,El pedido llegó dos horas tarde,problema de entrega o demora,problema de entrega o demora,0.744599,1.892159,True
5,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,No había stock y nadie sabía decirme cuándo re...,problema de stock o producto faltante,problema de stock o producto faltante,0.491183,2.133577,True
6,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,El pedido llegó tarde y además faltaba un prod...,problema de entrega o demora,problema de entrega o demora,0.962605,2.151772,True
7,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,pipeline,"Me cobraron bien, pero la atención fue pésima",problema de atención al cliente,problema de atención al cliente,0.865618,2.060109,True


## 5.3 Comparación de resultados con `pipeline`

Se comparan ambos modelos según cantidad de aciertos, accuracy y tiempo promedio de inferencia. El objetivo es determinar si el modelo de mayor capacidad mejora suficientemente la calidad de clasificación como para justificar su mayor costo computacional.

In [22]:
resumen = pd.DataFrame({
    "modelo": ["MiniLM", "mDeBERTa"],
    "aciertos": [
        resultados_minilm["acierto"].sum(),
        resultados_mdeberta["acierto"].sum()
    ],
    "total": [
        len(resultados_minilm),
        len(resultados_mdeberta)
    ],
    "accuracy": [
        resultados_minilm["acierto"].mean(),
        resultados_mdeberta["acierto"].mean()
    ],
    "tiempo_promedio_seg": [
        resultados_minilm["tiempo_segundos"].mean(),
        resultados_mdeberta["tiempo_segundos"].mean()
    ]
})

resumen

,modelo,aciertos,total,accuracy,tiempo_promedio_seg
0,MiniLM,4,8,0.500,0.033562
1,mDeBERTa,7,8,0.875,2.103783


### Resultado de la comparación

mDeBERTa obtuvo 7 aciertos sobre 8 casos (87,5 %), mientras que MiniLM alcanzó 4 aciertos sobre 8 (50 %).

MiniLM fue considerablemente más rápido, con un tiempo promedio de inferencia cercano a 0,03 segundos por mensaje. mDeBERTa tuvo un tiempo promedio aproximado de 2,07 segundos.

Por lo tanto, los resultados muestran un compromiso claro entre velocidad y calidad de clasificación: MiniLM prioriza eficiencia, mientras que mDeBERTa ofrece una mejora importante en la precisión sobre los mensajes evaluados.

Para la aplicación final, mDeBERTa aparece como el candidato principal si se prioriza la calidad de clasificación, aunque su mayor costo computacional deberá considerarse al momento del despliegue.

## 6. Prueba con `InferenceClient`

Luego se repetirá la comparación usando `InferenceClient`, manteniendo los mismos textos y etiquetas.

## 6. Evaluación con `InferenceClient`

Se repite la evaluación utilizando `InferenceClient`, manteniendo los mismos modelos, mensajes y categorías utilizados con `pipeline`.

A diferencia de `pipeline`, el modelo no se ejecuta en la computadora local, sino en la infraestructura del proveedor de inferencia. El objetivo es comparar no solo la calidad de clasificación, sino también la diferencia entre inferencia local y remota.

In [23]:
import os
import time
import pandas as pd

from getpass import getpass
from huggingface_hub import InferenceClient


In [25]:
os.environ["HF_TOKEN"] = getpass("Token de Hugging Face: ").strip()
print("Token cargado en memoria")

Token cargado en memoria


In [26]:
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"]
)

### 6.1 Función de evaluación con `InferenceClient`

Se utilizará la misma configuración de categorías y casos de prueba empleada con `pipeline`, para que la comparación entre inferencia local y remota sea consistente.

In [27]:
def evaluar_modelo_inference_client(nombre_modelo, casos, categorias):
    resultados = []

    for texto, categoria_esperada in casos:
        inicio = time.perf_counter()

        salida = client.zero_shot_classification(
            texto,
            candidate_labels=categorias,
            model=nombre_modelo,
            hypothesis_template="Este mensaje trata sobre {}."
        )

        duracion = time.perf_counter() - inicio

        categoria_predicha = salida[0].label
        score = salida[0].score

        resultados.append({
            "modelo": nombre_modelo,
            "metodo": "InferenceClient",
            "texto": texto,
            "esperada": categoria_esperada,
            "predicha": categoria_predicha,
            "score": score,
            "tiempo_segundos": duracion,
            "acierto": categoria_predicha == categoria_esperada
        })

    return pd.DataFrame(resultados)

In [ ]:
resultados_minilm_api = evaluar_modelo_inference_client(
    MODELO_MINILM,
    CASOS_PRUEBA,
    CATEGORIAS
)

resultados_minilm_api

,modelo,metodo,texto,esperada,predicha,score,tiempo_segundos,acierto
0,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,Me cobraron dos veces el mismo producto,problema de precio o facturación,problema de precio o facturación,0.427221,5.402707,True
1,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,No había leche ni yogur en la góndola,problema de stock o producto faltante,problema de stock o producto faltante,0.339001,0.273837,True
2,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,El pollo tenía mal olor cuando abrí el paquete,problema de calidad del producto,problema de entrega o demora,0.462015,0.265100,False
3,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,La persona que me atendió resolvió mi problema...,problema de atención al cliente,problema de atención al cliente,0.421866,0.294516,True
4,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,El pedido llegó dos horas tarde,problema de entrega o demora,problema de atención al cliente,0.437472,0.260992,False
5,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,No había stock y nadie sabía decirme cuándo re...,problema de stock o producto faltante,problema de stock o producto faltante,0.415952,0.265859,True
6,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,El pedido llegó tarde y además faltaba un prod...,problema de entrega o demora,problema de stock o producto faltante,0.389473,0.245845,False
7,MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli,InferenceClient,"Me cobraron bien, pero la atención fue pésima",problema de atención al cliente,problema de precio o facturación,0.276241,0.258823,False


### 6.2 Evaluación remota de mDeBERTa

Se repite la evaluación con mDeBERTa mediante `InferenceClient`, utilizando exactamente los mismos mensajes, categorías e `hypothesis_template` empleados en las pruebas anteriores.

In [ ]:
resultados_mdeberta_api = evaluar_modelo_inference_client(
    MODELO_MDEBERTA,
    CASOS_PRUEBA,
    CATEGORIAS
)

resultados_mdeberta_api

,modelo,metodo,texto,esperada,predicha,score,tiempo_segundos,acierto
0,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,Me cobraron dos veces el mismo producto,problema de precio o facturación,problema de precio o facturación,0.745498,4.725034,True
1,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,No había leche ni yogur en la góndola,problema de stock o producto faltante,problema de calidad del producto,0.258687,0.487482,False
2,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,El pollo tenía mal olor cuando abrí el paquete,problema de calidad del producto,problema de calidad del producto,0.975235,0.510933,True
3,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,La persona que me atendió resolvió mi problema...,problema de atención al cliente,problema de atención al cliente,0.998900,0.410061,True
4,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,El pedido llegó dos horas tarde,problema de entrega o demora,problema de entrega o demora,0.739664,0.397800,True
5,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,No había stock y nadie sabía decirme cuándo re...,problema de stock o producto faltante,problema de stock o producto faltante,0.482835,0.403039,True
6,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,El pedido llegó tarde y además faltaba un prod...,problema de entrega o demora,problema de entrega o demora,0.964507,0.772220,True
7,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,InferenceClient,"Me cobraron bien, pero la atención fue pésima",problema de atención al cliente,problema de atención al cliente,0.865617,0.461789,True


### 6.2.1 ¿Qué devuelve `InferenceClient`?

`InferenceClient.zero_shot_classification()` devuelve una lista de objetos de clasificación. Cada elemento expone la etiqueta y el puntaje como **atributos**.

La celda siguiente permite comparar esta estructura con el diccionario devuelto por `pipeline`.

In [ ]:
salida_api_tipo = client.zero_shot_classification(
    "Me cobraron dos veces el mismo producto",
    candidate_labels=CATEGORIAS,
    model=MODELO_MDEBERTA,
    hypothesis_template="Este mensaje trata sobre {}."
)

print("Tipo de salida:", type(salida_api_tipo))
print("Tipo del primer elemento:", type(salida_api_tipo[0]))
print("Etiqueta principal:", salida_api_tipo[0].label)
print("Score principal:", salida_api_tipo[0].score)

**Lectura del valor de retorno**

- La respuesta completa es una lista.
- Cada elemento representa una clasificación devuelta por el cliente de Hugging Face.
- La etiqueta se consulta con `salida_api_tipo[0].label`.
- El puntaje se consulta con `salida_api_tipo[0].score`.
- A diferencia de `pipeline`, aquí se accede a los datos mediante **atributos del objeto**.

Esta diferencia es importante porque el contenido semántico es equivalente, pero la estructura de retorno cambia según la interfaz utilizada.

## 6.3 Comparación global: modelo y método de inferencia

Se comparan los dos modelos utilizando tanto `pipeline` como `InferenceClient`. Esto permite separar dos decisiones distintas: qué modelo ofrece mejor calidad y dónde conviene ejecutar la inferencia.

In [ ]:
comparacion_global = pd.DataFrame({
    "modelo": [
        "MiniLM",
        "MiniLM",
        "mDeBERTa",
        "mDeBERTa"
    ],
    "metodo": [
        "pipeline",
        "InferenceClient",
        "pipeline",
        "InferenceClient"
    ],
    "aciertos": [
        resultados_minilm["acierto"].sum(),
        resultados_minilm_api["acierto"].sum(),
        resultados_mdeberta["acierto"].sum(),
        resultados_mdeberta_api["acierto"].sum()
    ],
    "total": [
        len(resultados_minilm),
        len(resultados_minilm_api),
        len(resultados_mdeberta),
        len(resultados_mdeberta_api)
    ],
    "accuracy": [
        resultados_minilm["acierto"].mean(),
        resultados_minilm_api["acierto"].mean(),
        resultados_mdeberta["acierto"].mean(),
        resultados_mdeberta_api["acierto"].mean()
    ],
    "tiempo_promedio_seg": [
        resultados_minilm["tiempo_segundos"].mean(),
        resultados_minilm_api["tiempo_segundos"].mean(),
        resultados_mdeberta["tiempo_segundos"].mean(),
        resultados_mdeberta_api["tiempo_segundos"].mean()
    ]
})

comparacion_global

,modelo,metodo,aciertos,total,accuracy,tiempo_promedio_seg
0,MiniLM,pipeline,4,8,0.500,0.033003
1,MiniLM,InferenceClient,4,8,0.500,0.908460
2,mDeBERTa,pipeline,7,8,0.875,2.066509
3,mDeBERTa,InferenceClient,7,8,0.875,1.021045


## 6.4 Conclusión de la comparación

Los resultados muestran que la elección del modelo tuvo un impacto mucho mayor que la elección del método de inferencia.

MiniLM alcanzó una accuracy del 50 % tanto con `pipeline` como con `InferenceClient`, mientras que mDeBERTa alcanzó 87,5 % en ambos casos. Esto confirma que cambiar entre ejecución local y remota no modifica la capacidad predictiva del modelo cuando se mantienen iguales el modelo, los textos, las categorías y la plantilla de hipótesis.

En cuanto al rendimiento, MiniLM fue el modelo más rápido en ejecución local. Sin embargo, mDeBERTa mediante `InferenceClient` presentó un tiempo promedio inferior al de su ejecución local en esta prueba.

Por este motivo, se selecciona **mDeBERTa** como modelo final por su mejor calidad de clasificación, y **InferenceClient** como estrategia principal para la aplicación publicada, ya que permite ejecutar un modelo más pesado sin cargar sus pesos en el servidor donde vive la interfaz.

## 7. Comparación de resultados

Se compararon los dos modelos utilizando tanto `pipeline` como `InferenceClient`, manteniendo constantes los mismos mensajes, categorías e `hypothesis_template`.

| Modelo | Método | Aciertos | Total | Accuracy | Tiempo promedio (s) |
|---|---|---:|---:|---:|---:|
| MiniLM | pipeline | 4 | 8 | 0.500 | 0.033 |
| MiniLM | InferenceClient | 4 | 8 | 0.500 | 0.908 |
| mDeBERTa | pipeline | 7 | 8 | 0.875 | 2.067 |
| mDeBERTa | InferenceClient | 7 | 8 | 0.875 | 1.021 |

Los resultados muestran que la elección del modelo tuvo un impacto mayor que el método de inferencia.

MiniLM obtuvo una accuracy del 50 % tanto en ejecución local como remota. mDeBERTa alcanzó 87,5 % en ambos casos. Esto indica que, al mantener el mismo modelo y la misma configuración, cambiar entre `pipeline` e `InferenceClient` no modifica sustancialmente la capacidad predictiva.

La principal diferencia entre ambos métodos estuvo en el lugar de ejecución y en los tiempos de respuesta. `pipeline` ejecuta el modelo localmente, mientras que `InferenceClient` delega la inferencia al proveedor remoto.

En esta prueba, mDeBERTa mediante `InferenceClient` tuvo un tiempo promedio menor que su ejecución local con `pipeline`, aunque la inferencia remota depende de conexión a internet, disponibilidad del proveedor y posibles tiempos de inicialización.

## 8. Selección del modelo final

Se selecciona `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` como modelo final para la aplicación.

La decisión se basa principalmente en su mejor desempeño sobre el conjunto de prueba: alcanzó 7 aciertos sobre 8 casos, equivalente a una accuracy del 87,5 %, frente al 50 % obtenido por MiniLM.

Para el despliegue se utilizará `InferenceClient`, ya que permite ejecutar el modelo de forma remota sin descargar ni cargar sus pesos en el servidor donde se aloja la interfaz. Esto reduce los requerimientos de memoria del despliegue y resulta conveniente para publicar la aplicación en infraestructura con recursos limitados.

La elección final prioriza, por lo tanto, la calidad de clasificación y la facilidad de despliegue por encima de la velocidad de inferencia local.

## 9. Estado final del proyecto

La comparación de modelos fue completada y se seleccionó `mDeBERTa` como modelo final.

La aplicación final:

- fue desarrollada con **Gradio**;
- utiliza **InferenceClient** para realizar la inferencia de forma remota;
- usa el modelo `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`;
- fue probada localmente;
- fue publicada en **Render**;
- utiliza la variable de entorno `HF_TOKEN` para no exponer credenciales;
- cuenta con `requirements.txt` y `README.md` reproducibles.

**App publicada:** https://seminario-actualizacion-2026.onrender.com

**Carpeta del TP:** https://github.com/ferfava/Seminario_Actualizacion_2026/tree/main/TP_Integrador

Como limitación principal, los mensajes ambiguos o con más de un motivo pueden producir clasificaciones discutibles. En una versión de producto sería conveniente ampliar el conjunto de evaluación y considerar clasificación multi-label cuando el caso de negocio lo requiera.